# 📈 Forecasting de Ventas 2025

Este notebook está diseñado para realizar el forecasting de ventas utilizando los datos de inferencia para 2025. A continuación, se importan las librerías necesarias y se carga el archivo de inferencia.

In [1]:
# 🤖 Importación de librerías principales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import streamlit as st
import holidays

In [13]:
# Cargar el archivo de inferencia en un DataFrame
inferencia_path = '../data/raw/inferencia/ventas_2025_inferencia.csv'
inferencia_df = pd.read_csv(inferencia_path)

# Mostrar las primeras filas para verificar la carga
inferencia_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


## Preparación de inferencia_df para el modelo de forecasting
A continuación se aplican todas las transformaciones, ingeniería de variables y codificaciones necesarias para que inferencia_df tenga la misma estructura y variables que el dataframe df usado en el entrenamiento.

In [14]:
# --- Transformaciones para preparar inferencia_df igual que df ---
import holidays

# 1. Asegurar tipo datetime en 'fecha'
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])

# 2. Variables temporales y de calendario
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.dayofweek  # 0=Lunes, 6=Domingo
inferencia_df['nombre_dia_semana'] = inferencia_df['fecha'].dt.day_name()
inferencia_df['semana_año'] = inferencia_df['fecha'].dt.isocalendar().week
inferencia_df['es_fin_de_semana'] = inferencia_df['dia_semana'].isin([5,6])

# Festivos en España para los años de inferencia
years_inf = inferencia_df['año'].unique()
festivos_es = holidays.country_holidays('ES', years=years_inf)
inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(festivos_es)

# Black Friday (último viernes de noviembre)
def es_black_friday(fecha):
    if fecha.month == 11 and fecha.weekday() == 4:
        ult_viernes = max([d for d in pd.date_range(start=fecha.replace(day=1), end=fecha.replace(day=30)) if d.weekday() == 4])
        return fecha == ult_viernes
    return False
inferencia_df['es_Black_Friday'] = inferencia_df['fecha'].apply(es_black_friday)

# Cyber Monday (primer lunes después de Black Friday)
def es_cyber_monday(fecha):
    if fecha.month == 11 or fecha.month == 12:
        year = fecha.year
        nov = pd.date_range(start=f'{year}-11-01', end=f'{year}-11-30', freq='D')
        fridays = nov[nov.weekday == 4]
        black_friday = fridays[-1]
        cyber_monday = black_friday + pd.Timedelta(days=3)
        return fecha == cyber_monday
    return False
inferencia_df['es_Cyber_Monday'] = inferencia_df['fecha'].apply(es_cyber_monday)

# Variables adicionales
inferencia_df['inicio_mes'] = inferencia_df['fecha'].dt.is_month_start
inferencia_df['fin_mes'] = inferencia_df['fecha'].dt.is_month_end
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['es_verano'] = inferencia_df['mes'].isin([6,7,8,9])
inferencia_df['es_navidad'] = inferencia_df['fecha'].dt.month.isin([12]) & (inferencia_df['fecha'].dt.day >= 15)

inferencia_df.head()

C:\Users\Usuario\AppData\Local\Temp\ipykernel_23376\1257097670.py:19: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(festivos_es)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,semana_año,es_fin_de_semana,es_festivo,es_Black_Friday,es_Cyber_Monday,inicio_mes,fin_mes,trimestre,es_verano,es_navidad
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,...,43,True,False,False,False,False,False,4,False,False
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,...,43,True,False,False,False,False,False,4,False,False
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,...,43,True,False,False,False,False,False,4,False,False
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,...,43,True,False,False,False,False,False,4,False,False
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,...,43,True,False,False,False,False,False,4,False,False


In [18]:
# 3. Crear lags de unidades_vendidas (1 a 7) y media móvil de 7 días, por año
lags = range(1, 8)
for lag in lags:
    inferencia_df[f'unidades_vendidas_lag{lag}'] = (
        inferencia_df.sort_values(['año', 'fecha'])
            .groupby(['año'])['unidades_vendidas']
            .shift(lag)
    )

# Media móvil de 7 días (incluye el día actual y los 6 anteriores), solo por año
inferencia_df['unidades_vendidas_mm7'] = (
    inferencia_df.sort_values(['año', 'fecha'])
        .groupby(['año'])['unidades_vendidas']
        .transform(lambda x: x.rolling(window=7, min_periods=7).mean())
)

cols_lag_mm = [f'unidades_vendidas_lag{lag}' for lag in lags] + ['unidades_vendidas_mm7']

# Eliminar registros de octubre (mes == 10)
inferencia_df = inferencia_df[inferencia_df['mes'] != 10].copy()

# No eliminar registros con nulos en los lags para noviembre, para poder ver todos los días aunque haya NaN
inferencia_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,es_verano,es_navidad,unidades_vendidas_lag1,unidades_vendidas_lag2,unidades_vendidas_lag3,unidades_vendidas_lag4,unidades_vendidas_lag5,unidades_vendidas_lag6,unidades_vendidas_lag7,unidades_vendidas_mm7
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
169,2025-11-01,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,NaN,135.00,NaN,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
170,2025-11-01,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,NaN,86.39,NaN,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
171,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,NaN,74.09,NaN,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
172,2025-11-01,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,NaN,34.76,NaN,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# 4. Variable de descuento porcentaje
inferencia_df['descuento_porcentaje'] = (inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base'] * 100

# 5. Calcular precio promedio de la competencia y ratio_precio
competidores = ['Amazon', 'Decathlon', 'Deporvillage']
inferencia_df['precio_competencia'] = inferencia_df[competidores].mean(axis=1)
inferencia_df['ratio_precio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']

# Eliminar columnas de los competidores
del_cols = [c for c in competidores if c in inferencia_df.columns]
inferencia_df = inferencia_df.drop(columns=del_cols)

inferencia_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,unidades_vendidas_lag2,unidades_vendidas_lag3,unidades_vendidas_lag4,unidades_vendidas_lag5,unidades_vendidas_lag6,unidades_vendidas_lag7,unidades_vendidas_mm7,descuento_porcentaje,precio_competencia,ratio_precio
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,100.610000,1.143028
169,2025-11-01,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,NaN,135.00,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,119.080000,1.133692
170,2025-11-01,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,NaN,86.39,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.635294,82.690000,1.044745
171,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,NaN,74.09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.213333,76.126667,0.973246
172,2025-11-01,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,NaN,34.76,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.685714,35.183333,0.987968


In [23]:
# 6. One Hot Encoding de nombre, categoría y subcategoría (con sufijo _h)
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']

one_hot = pd.get_dummies(inferencia_df[['nombre_h', 'categoria_h', 'subcategoria_h']], prefix=['nombre_h', 'categoria_h', 'subcategoria_h'])
inferencia_df = pd.concat([inferencia_df, one_hot], axis=1)

inferencia_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,False,False,False,False,False,False,True,False
169,2025-11-01,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,NaN,135.00,NaN,...,False,False,False,False,False,False,False,False,True,False
170,2025-11-01,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,NaN,86.39,NaN,...,False,False,False,False,False,False,False,False,True,False
171,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,NaN,74.09,NaN,...,False,False,False,False,False,False,False,False,True,False
172,2025-11-01,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,NaN,34.76,NaN,...,False,False,False,False,False,False,False,True,False,False


In [25]:
# Guardar inferencia_df transformado en data/processed
inferencia_df.to_csv('../data/processed/inferencia_df_transformado.csv', index=False)
print('Archivo guardado en ../data/processed/inferencia_df_transformado.csv')

Archivo guardado en ../data/processed/inferencia_df_transformado.csv


In [27]:
inferencia_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'año', 'mes', 'dia_mes', 'dia_semana', 'nombre_dia_semana',
       'semana_año', 'es_fin_de_semana', 'es_festivo', 'es_Black_Friday',
       'es_Cyber_Monday', 'inicio_mes', 'fin_mes', 'trimestre', 'es_verano',
       'es_navidad', 'unidades_vendidas_lag1', 'unidades_vendidas_lag2',
       'unidades_vendidas_lag3', 'unidades_vendidas_lag4',
       'unidades_vendidas_lag5', 'unidades_vendidas_lag6',
       'unidades_vendidas_lag7', 'unidades_vendidas_mm7',
       'descuento_porcentaje', 'precio_competencia', 'ratio_precio',
       'nombre_h', 'categoria_h', 'subcategoria_h',
       'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23',
       'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552',
       'nombre_h_Columbia Silver Ridge',
       'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Dom